# Load & Push

In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import datasets
from google.colab import drive

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

In [45]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="rebalance_procedure_human_factor.csv", repo_type="dataset",local_dir=".")
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

train_ds = datasets.load_dataset("sookiemonster/asrs-narratives", split="train")
valid_ds = datasets.load_dataset("sookiemonster/asrs-narratives", split="validation")
test_ds = datasets.load_dataset("sookiemonster/asrs-narratives", split="test")

In [11]:
rebalance_file = "rebalance_procedure_human_factor.csv"
df = pd.read_csv(rebalance_file, index_col=0)
df = df.sample(random_state=42, frac=1.0)
df

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
acn,,,
1350280,procedure,I observed the Local Controller clear a helico...,I observed helicopters landing and departing t...
1757607,humanfactors,I usually don't need to worry about landing cu...,NaN
1269800,procedure,Aircraft X on a STAAV6 departure; had departe...,NaN
1757028,humanfactors,[I was] advised by C Flight Attendant to do a ...,As I was cleaning the cabin during deplaning i...
1497666,procedure,We were on the ILS [approach]. Weather conditi...,Arrival and approach were in accordance with c...
...,...,...,...
1672454,procedure,Weather broken at 3500 ft. at ZZZ. ZZZ FCT (Fe...,NaN
1745783,humanfactors,I was the Captain (CA) and Pilot Flying (PF). ...,NaN
1746537,humanfactors,Working Radar; Handoff; and another sector com...,NaN


In [40]:
def parse_narrative_list(narratives:list[str]):
  return "\n".join([f"Narrative {idx+1} - '{narratives[idx]}'" for idx in range(len(narratives)) if pd.notna(narratives[idx])])

def structure_raw_df(df:pd.DataFrame):
  raw = df[['Report_1_Narrative', 'Report_2_Narrative']].copy().apply(
      lambda x: parse_narrative_list(x.to_list()),
      axis=1
  )
  narratives =  pd.DataFrame(raw, columns=['text'])
  narratives = narratives.join(df.Assessments_Primary_Problem)
  narratives = narratives.rename({'Assessments_Primary_Problem' : 'label'}, axis=1)
  return narratives

augment_df = structure_raw_df(df)
augment_df.head()

,text,label
acn,,
1350280,Narrative 1 - 'I observed the Local Controller...,procedure
1757607,Narrative 1 - 'I usually don't need to worry a...,humanfactors
1269800,Narrative 1 - 'Aircraft X on a STAAV6 departu...,procedure
1757028,Narrative 1 - '[I was] advised by C Flight Att...,humanfactors
1497666,Narrative 1 - 'We were on the ILS [approach]. ...,procedure


In [41]:
train_ds.features

{'acn': Value('int64'),
 'text': Value('string'),
 'label': ClassLabel(names=['aircraft', 'airport', 'airspacestructure', 'ambiguous', 'atcequipment/navfacility/buildings', 'chartorpublication', 'companypolicy', 'environment-nonweatherrelated', 'equipment/tooling', 'humanfactors', 'incorrect/notinstalled/unavailablepart', 'logbookentry', 'manuals', 'mel', 'procedure', 'softwareandautomation', 'staffing', 'weather'])}

In [46]:
from datasets import Dataset, concatenate_datasets, DatasetDict

augment_dataset = Dataset.from_pandas(augment_df)
augment_dataset = augment_dataset.cast_column('label', train_ds.features['label'])
augment_dataset.features

Casting the dataset:   0%|          | 0/6834 [00:00<?, ? examples/s]

{'text': Value('string'),
 'label': ClassLabel(names=['aircraft', 'airport', 'airspacestructure', 'ambiguous', 'atcequipment/navfacility/buildings', 'chartorpublication', 'companypolicy', 'environment-nonweatherrelated', 'equipment/tooling', 'humanfactors', 'incorrect/notinstalled/unavailablepart', 'logbookentry', 'manuals', 'mel', 'procedure', 'softwareandautomation', 'staffing', 'weather']),
 'acn': Value('int64')}

In [47]:
updated_train = concatenate_datasets([train_ds, augment_dataset])

updated_ds = DatasetDict({
    "train": updated_train,
    "validation": valid_ds,
    "test": test_ds
})

updated_ds

DatasetDict({
    train: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 17194
    })
    validation: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 4441
    })
    test: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 9868
    })
})

In [48]:
updated_ds.push_to_hub("sookiemonster/asrs-narratives-rebalance")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/18 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 16.7MB / 16.7MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  87%|########6 | 8.13MB / 9.38MB            

README.md:   0%|          | 0.00/935 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/sookiemonster/asrs-narratives-rebalance/commit/cd67a0f08169b13707f1fd67f6addd165ee780e3', commit_message='Upload dataset', commit_description='', oid='cd67a0f08169b13707f1fd67f6addd165ee780e3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sookiemonster/asrs-narratives-rebalance', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sookiemonster/asrs-narratives-rebalance'), pr_revision=None, pr_num=None)